# Fly Studio Research Execution

This is the single Colab notebook for running the real research workflow. It uses the existing repository scripts, does not fabricate data, and stops immediately when a gate fails.

## Section 1 - Environment

Clone the repository into a clean Colab workspace. The clone step is idempotent so rerunning the notebook does not create a nested checkout.

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/TanVi3001/drosophila-pd-flygym.git'
COLAB_ROOT = Path.cwd().resolve()
REPO_ROOT = COLAB_ROOT / 'drosophila-pd-flygym'
if COLAB_ROOT.name == 'drosophila-pd-flygym' and (COLAB_ROOT / '.git').is_dir():
    REPO_ROOT = COLAB_ROOT

# Equivalent shell commands: !git clone ... and %cd /content/drosophila-pd-flygym
if REPO_ROOT.exists() and not (REPO_ROOT / '.git').is_dir():
    raise RuntimeError(f'Expected a git checkout at {REPO_ROOT}, but the path is not a repository.')
if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
SOURCE_ROOT = REPO_ROOT / 'src'
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

def run_checked(command, *, cwd=REPO_ROOT):
    print('$', ' '.join(str(item) for item in command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)

print(f'Repository: {REPO_ROOT}')


### Python version gate

The pinned scientific runtime is Python 3.12.x. This cell fails before installation if the Colab runtime is incompatible.

In [ ]:
python_version = '.'.join(str(part) for part in sys.version_info[:3])
print(f'Python: {python_version}')
if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f'Unsupported Python {python_version}. Restart Colab with Python 3.12.x.')


### Install the declared package extras

Equivalent command: python -m pip install -e .[simulation,test]. Dependencies are installed from pyproject.toml; no notebook-specific workaround is used.

In [ ]:
run_checked([sys.executable, '-m', 'pip', 'install', '-e', '.[simulation,test]'])


### Runtime gate

Equivalent command: python scripts/check_runtime.py. A failed runtime check stops the notebook; no simulation or dataset is started.

In [ ]:
runtime_result = subprocess.run([sys.executable, 'scripts/check_runtime.py'], cwd=REPO_ROOT, check=False)
if runtime_result.returncode != 0:
    raise RuntimeError('Runtime gate failed. Install the pinned Python 3.12 FlyGym/MuJoCo runtime and rerun this notebook.')
print('Runtime gate: PASS')


## Section 2 - Smoke Test

Run the existing demo, then use the existing analysis CLI to complete the smoke artifact set. This does not create synthetic data.

In [ ]:
SMOKE_DATASET = REPO_ROOT / 'datasets' / 'healthy' / 'Healthy_001'
run_checked([sys.executable, 'scripts/run_demo.py', '--steps', '100'])
run_checked([
    sys.executable,
    'scripts/analyze_rollout.py',
    '--dataset',
    str(SMOKE_DATASET),
    '--output',
    str(SMOKE_DATASET),
])

smoke_required = [
    SMOKE_DATASET / 'rollout.json',
    SMOKE_DATASET / 'rollout.npz',
    SMOKE_DATASET / 'manifest.json',
    SMOKE_DATASET / 'metadata.json',
    SMOKE_DATASET / 'viewer_pose.json',
    SMOKE_DATASET / 'metrics' / 'metrics.json',
    SMOKE_DATASET / 'report' / 'summary.md',
]
smoke_missing = [str(path) for path in smoke_required if not path.is_file()]
if smoke_missing:
    raise RuntimeError('Smoke test artifact check failed: ' + ', '.join(smoke_missing))
print('Smoke test: PASS')


## Section 3 - Dataset Generation

The existing generator produces 20 datasets per configured group, for 80 total. It resumes complete datasets and stops on failed real simulation or artifact validation.

In [ ]:
DATASET_COUNT = 20
STEPS = 100
GENERATION_OUTPUT = REPO_ROOT / 'results' / 'research_dataset_generation'
print(f'DATASET_COUNT={DATASET_COUNT}; STEPS={STEPS}; expected total={DATASET_COUNT * 4}')


In [ ]:
run_checked([
    sys.executable,
    'scripts/generate_research_dataset.py',
    '--count',
    str(DATASET_COUNT),
    '--steps',
    str(STEPS),
])

generation_summary_path = GENERATION_OUTPUT / 'generation_summary.json'
if not generation_summary_path.is_file():
    raise RuntimeError(f'Missing generation summary: {generation_summary_path}')
generation_summary = json.loads(generation_summary_path.read_text(encoding='utf-8'))
generation_counts = generation_summary.get('counts', {})
failed_count = int(generation_counts.get('FAILED', 0))
completed_count = int(generation_counts.get('COMPLETED', 0))
skipped_count = int(generation_counts.get('SKIPPED', 0))
expected_count = DATASET_COUNT * 4
if failed_count or completed_count + skipped_count != expected_count:
    raise RuntimeError(f'Dataset generation gate failed: {generation_counts}')
print(f'Dataset generation: PASS ({completed_count} completed, {skipped_count} resumed)')


## Section 4 - Validation

Run the existing research validation workflow and verify its machine-readable boundary result. Dataset artifact completeness is also checked from the generator summary above.

In [ ]:
VALIDATION_OUTPUT = REPO_ROOT / 'results' / 'validation'
validation_result = subprocess.run([
    sys.executable,
    'scripts/validate_research_workflow.py',
    '--root',
    str(REPO_ROOT),
    '--output',
    str(VALIDATION_OUTPUT),
], cwd=REPO_ROOT, check=False)
if validation_result.returncode != 0:
    raise RuntimeError('Research validation command failed.')
validation_json = VALIDATION_OUTPUT / 'research_validation.json'
if not validation_json.is_file():
    raise RuntimeError(f'Missing validation report: {validation_json}')
validation_payload = json.loads(validation_json.read_text(encoding='utf-8'))
boundary = validation_payload.get('boundary', {})
if boundary and boundary.get('status') not in (None, 'PASS'):
    raise RuntimeError(f'Scientific boundary validation failed: {boundary}')
print('Validation: PASS')


## Section 5 - Research Pipeline

Run the existing orchestrator. The notebook checks every reported stage instead of trusting only the process exit code.

In [ ]:
run_checked([sys.executable, 'scripts/run_research_pipeline.py'])
research_status_path = REPO_ROOT / 'results' / 'research_status.json'
if not research_status_path.is_file():
    raise RuntimeError(f'Missing research status report: {research_status_path}')
research_status = json.loads(research_status_path.read_text(encoding='utf-8'))
required_stages = ('runtime', 'dataset', 'experiment', 'analysis', 'biomarkers', 'validation', 'release', 'publication')
stage_statuses = research_status.get('statuses', {})
failed_stages = {
    stage: stage_statuses.get(stage, {}).get('status')
    for stage in required_stages
    if stage_statuses.get(stage, {}).get('status') not in ('PASS', 'READY')
}
if failed_stages:
    raise RuntimeError(f'Research pipeline gate failed: {failed_stages}')

from IPython.display import Markdown, display
for report_name in ('research_status.md', 'progress_summary.md', 'final_execution_report.md'):
    report_path = REPO_ROOT / 'results' / report_name
    if not report_path.is_file():
        raise RuntimeError(f'Missing pipeline report: {report_path}')
    display(Markdown(f'## {report_name}\n\n' + report_path.read_text(encoding='utf-8')))
print('Research pipeline: PASS')


## Section 6 - Viewer Bundle

Build a static viewer bundle from the newest generated pose. The notebook does not open a browser or start a server.

In [ ]:
pose_candidates = sorted(
    (path for path in (REPO_ROOT / 'datasets').rglob('viewer_pose.json') if path.is_file()),
    key=lambda path: path.stat().st_mtime,
)
if not pose_candidates:
    raise RuntimeError('No viewer_pose.json was generated.')
LATEST_POSE = pose_candidates[-1]
VIEWER_BUNDLE = REPO_ROOT / 'results' / 'viewer_bundle.zip'
run_checked([
    sys.executable,
    'scripts/build_viewer_bundle.py',
    '--pose',
    str(LATEST_POSE),
    '--output',
    str(VIEWER_BUNDLE),
])
if not VIEWER_BUNDLE.is_file() or VIEWER_BUNDLE.stat().st_size == 0:
    raise RuntimeError(f'Viewer bundle was not created: {VIEWER_BUNDLE}')
print(f'Viewer bundle: PASS ({VIEWER_BUNDLE})')


## Section 7 - Download

Package the real datasets, results, and paper workspace. The archive is only created after all three source directories exist.

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

RESEARCH_OUTPUT = REPO_ROOT / 'research_output.zip'
archive_roots = ('datasets', 'results', 'paper')
missing_roots = [name for name in archive_roots if not (REPO_ROOT / name).is_dir()]
if missing_roots:
    raise RuntimeError('Cannot package missing output roots: ' + ', '.join(missing_roots))
if RESEARCH_OUTPUT.exists():
    RESEARCH_OUTPUT.unlink()
with ZipFile(RESEARCH_OUTPUT, 'w', compression=ZIP_DEFLATED) as archive:
    for root_name in archive_roots:
        root_path = REPO_ROOT / root_name
        for path in sorted(root_path.rglob('*')):
            if path.is_file():
                archive.write(path, path.relative_to(REPO_ROOT).as_posix())
if not RESEARCH_OUTPUT.is_file() or RESEARCH_OUTPUT.stat().st_size == 0:
    raise RuntimeError('research_output.zip was not created.')
print(f'Research output: PASS ({RESEARCH_OUTPUT.stat().st_size} bytes)')

from google.colab import files
files.download(str(RESEARCH_OUTPUT))


## Section 8 - Final Summary

In [ ]:
dataset_paths = sorted({path.parent for path in (REPO_ROOT / 'datasets').rglob('rollout.json') if path.is_file()})
rollout_count = len(dataset_paths)
viewer_pose_count = sum((path / 'viewer_pose.json').is_file() for path in dataset_paths)
biomarker_files = sorted(path for path in (REPO_ROOT / 'results').rglob('biomarkers.json') if path.is_file())
report_files = sorted(path for path in (REPO_ROOT / 'results').rglob('summary.md') if path.is_file())
runtime_text = '.'.join(str(part) for part in sys.version_info[:3])
final_statuses = research_status.get('statuses', {})
final_failures = {name: value.get('status') for name, value in final_statuses.items() if value.get('status') not in ('PASS', 'READY')}
if final_failures or viewer_pose_count != rollout_count or rollout_count != DATASET_COUNT * 4:
    raise RuntimeError(f'Final summary gate failed: failures={final_failures}, rollouts={rollout_count}, viewer_poses={viewer_pose_count}')
print('Research execution summary')
print(f'- datasets / rollouts: {rollout_count}')
print(f'- viewer poses: {viewer_pose_count}')
print(f'- biomarker reports: {len(biomarker_files)}')
print(f'- report summaries: {len(report_files)}')
print(f'- runtime: Python {runtime_text}')
print('- status: PASS')
print('Research execution completed successfully.')
